# 08 — Training Run Comparison

Load multiple training run bundles and compare them side-by-side.
Useful for hyperparameter sweeps and ablation studies.

**Input:** A list of `run_id` strings (from `outputs/training/` or unpacked Drive bundles).  
**Output:** A ranked table + side-by-side reward curves.

In [ ]:
import sys
from pathlib import Path

# Add src/ to Python path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

from optimized_llm_planning_memory.training.run_logger import load_episode_metrics, load_ppo_metrics, list_run_ids
from optimized_llm_planning_memory.training.run_manifest import load_manifest, list_manifests

OUTPUT_DIR = REPO_ROOT / 'outputs'
TRAINING_DIR = OUTPUT_DIR / 'training'

print(f'Training dir: {TRAINING_DIR}')
print(f'Available runs: {list_run_ids(TRAINING_DIR)}')

In [ ]:
# ── Configure which runs to compare ─────────────────────────────────────────
# Leave as None to compare ALL runs in TRAINING_DIR.
# Or specify a list: ['20260501_120000', '20260502_093000']

RUN_IDS_TO_COMPARE = None  # None = all runs

if RUN_IDS_TO_COMPARE is None:
    RUN_IDS_TO_COMPARE = list_run_ids(TRAINING_DIR)

print(f'Comparing {len(RUN_IDS_TO_COMPARE)} runs:')
for rid in RUN_IDS_TO_COMPARE:
    print(f'  {rid}')

In [ ]:
# ── Load episode metrics and manifests for all selected runs ─────────────────

run_data = {}
manifest_data = {}

for run_id in RUN_IDS_TO_COMPARE:
    ep_records = load_episode_metrics(run_id, TRAINING_DIR)
    ppo_records = load_ppo_metrics(run_id, TRAINING_DIR)
    manifest = load_manifest(run_id, TRAINING_DIR)

    if ep_records:
        run_data[run_id] = {
            'ep_df': pd.DataFrame([r.model_dump() for r in ep_records]),
            'ppo_df': pd.DataFrame([r.model_dump() for r in ppo_records]) if ppo_records else pd.DataFrame(),
        }
        manifest_data[run_id] = manifest
        print(f'  {run_id}: {len(ep_records)} episodes, {len(ppo_records)} PPO updates')
    else:
        print(f'  {run_id}: no data (skipping)')

print(f'\nLoaded {len(run_data)} runs with data.')

In [ ]:
# ── Summary table: one row per run ───────────────────────────────────────────

rows = []
for run_id, data in run_data.items():
    ep_df = data['ep_df']
    ppo_df = data['ppo_df']
    manifest = manifest_data.get(run_id)

    last_n = ep_df.tail(50)
    row = {
        'run_id': run_id,
        'compressor_type': manifest.compressor_type if manifest else '?',
        'run_name': manifest.run_name if manifest else '',
        'n_episodes': len(ep_df),
        'final_hard_constraint': last_n['hard_constraint_score'].mean() if 'hard_constraint_score' in last_n else float('nan'),
        'final_soft_constraint': last_n['soft_constraint_score'].mean() if 'soft_constraint_score' in last_n else float('nan'),
        'final_reward_mean': last_n['total_reward'].mean() if 'total_reward' in last_n else float('nan'),
        'final_tool_efficiency': last_n['tool_efficiency_score'].mean() if 'tool_efficiency_score' in last_n else float('nan'),
        'final_success_rate': last_n['success'].mean() if 'success' in last_n else float('nan'),
        'lr': manifest.ppo_hyperparams.get('learning_rate', '?') if manifest else '?',
        'clip_eps': manifest.ppo_hyperparams.get('clip_epsilon', '?') if manifest else '?',
        'ent_coef': manifest.ppo_hyperparams.get('ent_coef', '?') if manifest else '?',
    }
    rows.append(row)

summary_df = pd.DataFrame(rows).sort_values('final_hard_constraint', ascending=False)
print('=== Run Comparison Summary (sorted by final hard constraint score) ===')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
display(summary_df.round(3))

In [ ]:
# ── Side-by-side reward curves ────────────────────────────────────────────────

if not run_data:
    print('No run data to plot.')
else:
    colors = cm.tab10(np.linspace(0, 1, len(run_data)))
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Training Run Comparison', fontsize=14)

    metrics = [
        ('total_reward', 'Total Reward', axes[0, 0]),
        ('hard_constraint_score', 'Hard Constraint Score', axes[0, 1]),
        ('soft_constraint_score', 'Soft Constraint Score', axes[1, 0]),
        ('tool_efficiency_score', 'Tool Efficiency', axes[1, 1]),
    ]

    for col, title, ax in metrics:
        for (run_id, data), color in zip(run_data.items(), colors):
            ep_df = data['ep_df']
            if col not in ep_df.columns:
                continue
            label = run_id[-8:]  # last 8 chars of timestamp
            manifest = manifest_data.get(run_id)
            if manifest and manifest.run_name:
                label = manifest.run_name
            window = max(1, len(ep_df) // 20)
            ax.plot(ep_df[col].rolling(window, min_periods=1).mean().values,
                    color=color, linewidth=1.5, label=label)
        ax.set_title(title)
        ax.set_xlabel('Episode')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    out_path = OUTPUT_DIR / 'run_comparison.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_path}')

In [ ]:
# ── PPO convergence comparison ────────────────────────────────────────────────

ppo_runs = {rid: d for rid, d in run_data.items() if not d['ppo_df'].empty}

if ppo_runs:
    colors = cm.tab10(np.linspace(0, 1, len(ppo_runs)))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('PPO Convergence Diagnostics', fontsize=13)

    for (run_id, data), color in zip(ppo_runs.items(), colors):
        ppo_df = data['ppo_df']
        manifest = manifest_data.get(run_id)
        label = (manifest.run_name if manifest and manifest.run_name else run_id[-8:])

        for ax, col, title in [
            (axes[0], 'approx_kl', 'Approx KL (< 0.02 healthy)'),
            (axes[1], 'clip_fraction', 'Clip Fraction (< 0.2 healthy)'),
            (axes[2], 'explained_variance', 'Explained Variance (→ 1.0)'),
        ]:
            if col in ppo_df.columns:
                ax.plot(ppo_df[col].values, color=color, linewidth=1.2, label=label)
            ax.set_title(title, fontsize=10)
            ax.legend(fontsize=7)
            ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ppo_convergence_comparison.png', dpi=150)
    plt.show()
else:
    print('No PPO update data found in selected runs.')

In [ ]:
# ── Export summary table to CSV ───────────────────────────────────────────────

csv_path = OUTPUT_DIR / 'run_comparison_summary.csv'
summary_df.to_csv(csv_path, index=False)
print(f'Summary saved → {csv_path}')

# Highlight best run
if not summary_df.empty:
    best = summary_df.iloc[0]
    print(f'\n=== Best run by final hard constraint score ===')
    print(f'  run_id          : {best["run_id"]}')
    print(f'  run_name        : {best["run_name"]}')
    print(f'  compressor_type : {best["compressor_type"]}')
    print(f'  learning_rate   : {best["lr"]}')
    print(f'  clip_epsilon    : {best["clip_eps"]}')
    print(f'  ent_coef        : {best["ent_coef"]}')
    print(f'  hard_constraint : {best["final_hard_constraint"]:.3f}')
    print(f'  reward_mean     : {best["final_reward_mean"]:.3f}')